# Re-run: reliability (Paper II, RQ2)

This notebook recomputes the headline numbers of RQ2 from the committed outputs in the code package of
release `learning-graph-budget-2026`. It needs Python 3.9+ with numpy and the unzipped release (run it
from the release folder or its `notebooks/` folder, or set `RELEASE_DIR`).

**What it recomputes, and from what.** The split halves themselves read XES3G5M and EdNet KT1, which may
not be redistributed; `reproduce.py` reruns them for a reader who holds the corpora (steps `rq2-*`).
The release keeps, for each reading, the mean split-half correlation over the random splits, each
arm's κ and its deranged κ at each number of learners, the confusion counts of the simulated floor
world, and the number of false edges stability selection kept in **each** of the 20 simulated worlds.
From those the notebook recomputes:

- Table 3: the Spearman–Brown coefficient 2r/(1 + r) from the mean split-half correlation r, the same
  for the reach-partialled correlation, and the ceiling √SB;
- Table 5: each excess κ (κ minus the deranged κ), the rise from the smallest to the largest sample,
  and the saturating curve κ(n) = k_max·n/(n + n½) refitted by least squares, with the learners per
  half an excess of 0.60 needs;
- the floor world: the precision of the 2,507 asserted edges from the planted gates they recover;
- Table 7: the mean number of false edges over the 20 worlds of each kind, and its distance from the
  bound q²/((2τ − 1)p), which it recomputes for the real runs as well.

Bootstrap intervals are not redrawn here; their endpoints are the study's.

In [1]:
# Finding the published data. This cell is the same in every notebook of this report.
import gzip, json, os
from pathlib import Path

RELEASE_ID = "learning-graph-budget-2026"   # filled in by board/render_report.py
DATA_NAME = "per-case-model-outputs.jsonl"


def find_release_file(name=DATA_NAME):
    """Look beside this notebook, in RELEASE_DIR, in the folders above, and in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    places += [here, *here.parents]
    places += [p / "research/wiki/research/releases" / RELEASE_ID for p in [here, *here.parents]]
    for d in places:
        for n in (name, name + ".gz"):
            if (d / n).is_file():
                return d / n
    raise FileNotFoundError(
        f"Could not find {name} (or {name}.gz). Put it in the same folder as this notebook: download it "
        f"from the report ('Download the data file', beside each notebook) or take it from the release "
        f"directory. Or set the environment variable RELEASE_DIR to the folder that holds it.")


def per_case(runs=None):
    """Yield the per-case records of the published file, optionally only for the named runs."""
    path = find_release_file()
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            if r.get("record") == "case" and (runs is None or r["run"] in runs):
                yield r


def code_dir():
    """The release's code package (its code/ folder): in RELEASE_DIR, beside this notebook or in a folder
    above it (as when the notebook sits in the unzipped release's notebooks/ folder), in a folder named
    after the release there, or in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    for p in [here, *here.parents]:
        places += [p, p / RELEASE_ID, p / "research/wiki/research/releases" / RELEASE_ID]
    for d in places:
        if (d / "code" / "SHA256SUMS").is_file():
            return d / "code"
    raise FileNotFoundError(
        f"Could not find the code package of release {RELEASE_ID} (its code/ folder). Download the release "
        f"from the report ('Download the release', in the sidebar or beside each notebook), unzip it, and "
        f"run this notebook from the unzipped folder or its notebooks/ folder. Or set the environment "
        f"variable RELEASE_DIR to the unzipped {RELEASE_ID} folder.")


def find_code_file(relpath):
    """A file inside the release's code package, by its path there (e.g. studies/.../out/result.json)."""
    path = code_dir() / relpath
    if not path.is_file():
        raise FileNotFoundError(f"The code package at {code_dir()} has no {relpath}: is it release {RELEASE_ID}?")
    return path


def load_code_json(relpath):
    """A committed output of the code package, parsed."""
    return json.loads(find_code_file(relpath).read_text(encoding="utf-8"))


def shown(path):
    """A path as printed: relative to this notebook's folder, so an executed notebook names no machine's folders."""
    return os.path.relpath(path, Path.cwd())


def corpus_file(env_var, name):
    """A public corpus file the reader has downloaded: $env_var/name, or name beside this notebook."""
    for d in ([Path(os.environ[env_var])] if os.environ.get(env_var) else []) + [Path.cwd()]:
        if (d / name).is_file():
            return d / name
    raise FileNotFoundError(f"This notebook also needs {name} from the public corpus. Download the corpus "
                            f"(its location and hash are in the release file), then set {env_var} to its "
                            f"folder or put {name} beside this notebook.")


print("Code package:", shown(code_dir()))

Code package: code


In [2]:
import json
import numpy as np

sg = lambda x, d=3: f"{x:+.{d}f}".replace("-", "−")
split = load_code_json("studies/measurement/reliability/out/split_half.json")
print(f"Bar: {split['gate']:.2f}; {split['splits']} random splits")
sb = lambda r: 2 * r / (1 + r)
PRINTED_T3 = {  # (corpus, floor): (pairs, split-half, Spearman-Brown, reach-partialled SB, ceiling)
    ("xes3g5m", 5): (1261, "0.342", "0.510", "0.507", "0.714"), ("xes3g5m", 10): (1036, "0.392", "0.563", "0.560", "0.750"),
    ("xes3g5m", 20): (812, "0.474", "0.643", "0.640", "0.802"), ("ednet", 5): (4382, "0.051", "0.098", "0.096", "0.313"),
    ("ednet", 10): (3569, "0.067", "0.125", "0.122", "0.354"), ("ednet", 20): (2343, "0.073", "0.136", "0.127", "0.369")}
for row in split["rows"]:
    r = row["split_half"]
    got = (row["pairs_median"], f"{r:.3f}", f"{sb(r):.3f}", f"{sb(row['split_half_reach_partialled']):.3f}",
           f"{np.sqrt(sb(r)):.3f}")
    print(f"{row['corpus']:8s} floor {row['min_cell']:2d}: pairs {got[0]:5,d}  r {got[1]}  SB {got[2]}  reach-partialled {got[3]}  ceiling {got[4]}")
    assert got == PRINTED_T3[(row["corpus"], row["min_cell"])], row
    assert sb(r) < split["gate"]                       # no row reaches 0.80
best = max((sb(x["split_half"]), x["corpus"], x["min_cell"]) for x in split["rows"])
assert (f"{best[0]:.3f}", best[1], best[2]) == ("0.643", "xes3g5m", 20)
gap = max(abs(sb(x["split_half"]) - sb(x["split_half_reach_partialled"])) for x in split["rows"])
assert f"{gap:.3f}" == "0.009"                         # "changed the coefficient by at most 0.009"
print("Table 3 (30 values) matches the paper; no row reaches 0.80; partialling out reach moves SB by at most 0.009.")

Bar: 0.80; 12 random splits
xes3g5m  floor  5: pairs 1,261  r 0.342  SB 0.510  reach-partialled 0.507  ceiling 0.714
xes3g5m  floor 10: pairs 1,036  r 0.392  SB 0.563  reach-partialled 0.560  ceiling 0.750
xes3g5m  floor 20: pairs   812  r 0.474  SB 0.643  reach-partialled 0.640  ceiling 0.802
ednet    floor  5: pairs 4,382  r 0.051  SB 0.098  reach-partialled 0.096  ceiling 0.313
ednet    floor 10: pairs 3,569  r 0.067  SB 0.125  reach-partialled 0.122  ceiling 0.354
ednet    floor 20: pairs 2,343  r 0.073  SB 0.136  reach-partialled 0.127  ceiling 0.369
Table 3 (30 values) matches the paper; no row reaches 0.80; partialling out reach moves SB by at most 0.009.


## Agreement on which pairs to assert, against the number of learners (Table 5)

In [3]:
curves = {c: load_code_json(f"studies/measurement/reliability/out/raise_kappa_curve_{c}.json") for c in ("xes3g5m", "ednet")}
PRINTED_T5 = {"xes3g5m": [(1354, "+0.210", "+0.009", "+0.202"), (2709, "+0.306", "+0.014", "+0.291"),
                          (4516, "+0.376", "+0.009", "+0.367"), (6774, "+0.456", "+0.005", "+0.451"),
                          (9033, "+0.504", "+0.030", "+0.474")],
              "ednet": [(4500, "+0.071", "+0.016", "+0.055"), (9000, "+0.154", "+0.013", "+0.140"),
                        (15000, "+0.234", "+0.013", "+0.220"), (22500, "+0.325", "+0.015", "+0.310"),
                        (30000, "+0.400", "−0.004", "+0.403")]}
for c, cur in curves.items():
    pts = sorted(cur["points"].values(), key=lambda p: p["per_half"])
    got = [(p["per_half"], sg(p["kappa"]), sg(p["deranged"]), sg(p["kappa"] - p["deranged"])) for p in pts]
    for g in got:
        print(c, *g)
    assert got == PRINTED_T5[c], c
    rise = (pts[-1]["kappa"] - pts[-1]["deranged"]) - (pts[0]["kappa"] - pts[0]["deranged"])
    assert abs(rise - cur["excess_rise"]) < 1e-12
    assert pts[0]["excess_hi"] < pts[-1]["excess_lo"]     # the two intervals are disjoint, smallest below largest
    print(f"{c}: excess rose by {sg(rise)}, intervals disjoint -> {cur['verdict']}")
assert (sg(curves["xes3g5m"]["excess_rise"]), sg(curves["ednet"]["excess_rise"])) == ("+0.272", "+0.348")
der = [p["deranged"] for cur in curves.values() for p in cur["points"].values()]
assert (sg(min(der)), sg(max(der))) == ("−0.004", "+0.030")
print("Table 5 (30 values) and the rises (+0.272, +0.348) match; the deranged arm stays between −0.004 and +0.030.")

xes3g5m 1354 +0.210 +0.009 +0.202
xes3g5m 2709 +0.306 +0.014 +0.291
xes3g5m 4516 +0.376 +0.009 +0.367
xes3g5m 6774 +0.456 +0.005 +0.451
xes3g5m 9033 +0.504 +0.030 +0.474
xes3g5m: excess rose by +0.272, intervals disjoint -> SAMPLE IS THE CEILING
ednet 4500 +0.071 +0.016 +0.055
ednet 9000 +0.154 +0.013 +0.140
ednet 15000 +0.234 +0.013 +0.220
ednet 22500 +0.325 +0.015 +0.310
ednet 30000 +0.400 −0.004 +0.403
ednet: excess rose by +0.348, intervals disjoint -> SAMPLE IS THE CEILING
Table 5 (30 values) and the rises (+0.272, +0.348) match; the deranged arm stays between −0.004 and +0.030.


In [4]:
# The saturating curve, refitted to XES3G5M's five excess points by least squares. For a fixed n_half the
# best k_max is closed-form; n_half is found by a grid and golden-section search.
pts = sorted(curves["xes3g5m"]["points"].values(), key=lambda p: p["per_half"])
n = np.array([p["per_half"] for p in pts], float)
y = np.array([p["excess"] for p in pts], float)

def best_k(h):
    g = n / (n + h)
    k = (g @ y) / (g @ g)
    return k, ((y - k * g) ** 2).sum()

hs = np.exp(np.linspace(np.log(1), np.log(1e8), 4001))
h0 = hs[np.argmin([best_k(h)[1] for h in hs])]
a, b = np.log(h0) - 0.01, np.log(h0) + 0.01
phi = (np.sqrt(5) - 1) / 2
for _ in range(200):
    c1, c2 = b - phi * (b - a), a + phi * (b - a)
    a, b = (a, c2) if best_k(np.exp(c1))[1] < best_k(np.exp(c2))[1] else (c1, b)
nh = np.exp((a + b) / 2)
kmax = best_k(nh)[0]
need60 = 0.6 * nh / (kmax - 0.6)                      # solve k_max n / (n + n_half) = 0.60 for n
fit = curves["xes3g5m"]["fit"]
print(f"k_max {sg(kmax)} (study {sg(fit['k_max'])}), n_half {nh:,.0f}; an excess of 0.60 needs {need60:,.0f} learners per half")
assert sg(kmax) == sg(fit["k_max"]) == "+0.644" and abs(nh / fit["n_half"] - 1) < 1e-3
assert f"{need60:,.0f}" == "43,466"
ci = fit["k_max_ci"]
assert (sg(ci[0]), sg(ci[1])) == ("+0.584", "+0.722") and ci[1] < 0.80   # 0.80 lies above the interval
print("The ceiling +0.644 [+0.584, +0.722] and the 43,466 learners per half match; 0.80 is out of reach.")

k_max +0.644 (study +0.644), n_half 3,171; an excess of 0.60 needs 43,466 learners per half
The ceiling +0.644 [+0.584, +0.722] and the 43,466 learners per half match; 0.80 is out of reach.


## The floor world: reliable, and almost all of its edges false

In the simulated floor world with 32,000 learners, the estimated graph's ranking reached a
Spearman–Brown coefficient of +0.899. Its asserted edges recover a share (recall) of the 194 planted
gates; precision is the planted gates recovered over the edges asserted.

In [5]:
pt = next(p for p in load_code_json("studies/measurement/reliability/out/kappa_bar_floor.json")["points"] if p["n"] == 32000)
tp = round(pt["recall"] * pt["true_edges"])
assert tp / pt["true_edges"] == pt["recall"]
prec = tp / pt["edges"]
print(f"SB {sg(pt['sb'])}, kappa {sg(pt['kappa'])} (excess {sg(pt['kappa'] - pt['deranged_kappa'])}); {pt['edges']:,} edges asserted, "
      f"{tp} of the {pt['true_edges']} planted gates among them: precision {prec:.3f}")
assert abs(prec - pt["precision"]) < 1e-12
assert (sg(pt["sb"]), sg(pt["kappa"]), sg(pt["kappa"] - pt["deranged_kappa"]), f"{pt['edges']:,}", pt["true_edges"], f"{prec:.3f}") == \
    ("+0.899", "+0.646", "+0.646", "2,507", 194, "0.075")
assert pt["sb"] > 0.80

curve = load_code_json("studies/measurement/stability-selection/out/sim.json")["worlds"]["I0/0.1"]
rows = curve["curve_adopted"]["rows"]
precs = [r["precision"][0] for r in rows]
assert (f"{min(precs):.3f}", f"{max(precs):.3f}") == ("0.074", "0.095")
sel = next(r for r in curve["curve_selected"]["rows"] if r["n"] == 32000)
assert (f"{sel['precision'][0]:.3f}", f"{sel['sensitivity'][0]:.3f}") == ("0.956", "0.150")
print(f"Across 20 worlds the adopted rule's precision stays between {min(precs):.3f} and {max(precs):.3f} "
      f"from 500 to 32,000 learners; stability selection: precision {sel['precision'][0]:.3f} at sensitivity {sel['sensitivity'][0]:.3f}.")

SB +0.899, kappa +0.646 (excess +0.646); 2,507 edges asserted, 188 of the 194 planted gates among them: precision 0.075
Across 20 worlds the adopted rule's precision stays between 0.074 and 0.095 from 500 to 32,000 learners; stability selection: precision 0.956 at sensitivity 0.150.


## Whether the stability-selection bound holds (Table 7)

For each kind of world, the false edges kept at τ in each of the 20 worlds (`V_per_rep`), their mean,
and the mean minus the bound. The bound for *q* pairs selected per half-sample out of *p* candidates
is q²/((2τ − 1)p).

In [6]:
sim = load_code_json("studies/measurement/stability-selection/out/sim.json")["worlds"]
PRINTED_T7 = {"I0": ("1.20", "0.997", "+0.20", "2.25", "2.49"), "I": ("0.90", "0.997", "−0.10", "1.60", "2.49"),
              "C": ("0.95", "0.986", "−0.04", "3.45", "2.46"), "C0": ("0.70", "0.986", "−0.29", "2.00", "2.47")}
for kind, want in PRINTED_T7.items():
    w = sim[f"{kind}/0.1"]
    p75, p60 = w["bound"]["32000"], w["bound_tau"]["0.60"]
    v75, v60 = np.mean(p75["V_per_rep"]), np.mean(p60["V_per_rep"])
    assert len(p75["V_per_rep"]) == len(p60["V_per_rep"]) == 20
    got = (f"{v75:.2f}", f"{p75['mean_bound']:.3f}", sg(v75 - p75["mean_bound"], 2), f"{v60:.2f}", f"{p60['mean_bound']:.2f}")
    print(f"{kind:3s} tau 0.75: V {got[0]} bound {got[1]} V-bound {got[2]} [{p75['verdict']}]   tau 0.60: V {got[3]} bound {got[4]} [{p60['verdict']}]")
    assert got == want, (kind, got)
assert np.mean(sim["C/0.1"]["bound_tau"]["0.60"]["V_per_rep"]) > sim["C/0.1"]["bound_tau"]["0.60"]["mean_bound"]   # C breaks it at 0.60
assert all(abs(np.mean(sim[f"{k}/0.1"]["bound"]["32000"]["V_per_rep"]) - sim[f"{k}/0.1"]["bound"]["32000"]["mean_bound"]) < 0.3
           for k in PRINTED_T7)                           # "every world sat within 0.3 false edges of its bound"
q, p = 42, 3540
assert f"{q ** 2 / ((2 * 0.75 - 1) * p):.3f}" == "0.997"     # the simulated world's bound (q = 42, p = 3,540)
real = load_code_json("studies/measurement/stability-selection/out/real.json")
for c, want_bound in (("xes3g5m", "0.997"), ("ednet", "1.000")):
    r = real[c]["real"][0]
    bound = r["q"] ** 2 / ((2 * 0.75 - 1) * r["p"])
    print(f"{c}: q {r['q']}, p {r['p']:,} -> bound {bound:.3f}; selected at 0.75: {[x['0.75']['selected'] for x in real[c]['real']]}")
    assert f"{bound:.3f}" == want_bound and abs(bound - r["bounds"]["0.75"]) < 1e-12
assert [x["0.75"]["selected"] for x in real["xes3g5m"]["real"]] == [10, 9]
assert [x["0.75"]["selected"] for x in real["ednet"]["real"]] == [0, 0]
print("Table 7 and the bounds match the paper.")

I0  tau 0.75: V 1.20 bound 0.997 V-bound +0.20 [VIOLATED (marginal)]   tau 0.60: V 2.25 bound 2.49 [HONEST (marginal)]
I   tau 0.75: V 0.90 bound 0.997 V-bound −0.10 [HONEST (marginal)]   tau 0.60: V 1.60 bound 2.49 [HONEST (decisive)]
C   tau 0.75: V 0.95 bound 0.986 V-bound −0.04 [HONEST (marginal)]   tau 0.60: V 3.45 bound 2.46 [VIOLATED (decisive)]
C0  tau 0.75: V 0.70 bound 0.986 V-bound −0.29 [HONEST (marginal)]   tau 0.60: V 2.00 bound 2.47 [HONEST (marginal)]
xes3g5m: q 127, p 32,362 -> bound 0.997; selected at 0.75: [10, 9]
ednet: q 87, p 15,138 -> bound 1.000; selected at 0.75: [0, 0]
Table 7 and the bounds match the paper.
